In [0]:
from pyspark.sql import functions as F

# Read the Bronze Delta table
bronze_df = spark.table("workspace.default.bronze_wind_power")

# Display the schema
bronze_df.printSchema()

# Display a sample of the data
display(bronze_df.limit(10))

In [0]:
silver_df = bronze_df.withColumn(
    "measurement_timestamp",
    F.to_timestamp(
        F.concat_ws(" ", F.col("date"), F.col("time")),
        "yyyy-MM-dd HH-mm-ss"
    )
)

In [0]:
display(silver_df.limit(10))

In [0]:
# Count null values in every column
null_counts = silver_df.select([
    F.count(
        F.when(F.col(column).isNull(), 1)
    ).alias(column)
    for column in silver_df.columns
])
display(null_counts)
print(len(silver_df.columns))
print(silver_df.count())

In [0]:
# Define the target Silver table
silver_table = "workspace.default.silver_wind_power"

# Save the transformed DataFrame as a Delta table
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print(f"Silver table created successfully: {silver_table}")

In [0]:
# Read the persisted Silver table
silver_check_df = spark.table(
    "workspace.default.silver_wind_power"
)

# Validate row count and column count
print("Number of rows:", silver_check_df.count())
print("Number of columns:", len(silver_check_df.columns))

# Inspect the schema and sample records
silver_check_df.printSchema()

display(silver_check_df.limit(10))

In [0]:
# Load the Silver table for data quality validation

silver_df = spark.table("workspace.default.silver_wind_power")

print("Number of rows:", silver_df.count())
print("Number of columns:", len(silver_df.columns))

In [0]:
from pyspark.sql import functions as F

# Count records with a missing production identifier

missing_id_count = silver_df.filter(
    F.col("production_id").isNull()
).count()

print("DQ-01 - Missing production IDs:", missing_id_count)

if missing_id_count == 0:
    print("Result: PASS")
else:
    print("Result: FAIL")

In [0]:
# Detect duplicated production identifiers

duplicate_id_count = (
    silver_df
    .groupBy("production_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("DQ-02 - Duplicated production IDs:", duplicate_id_count)

if duplicate_id_count == 0:
    print("Result: PASS")
else:
    print("Result: FAIL")

In [0]:
# DQ-03: Check for missing turbine names

missing_turbine_count = silver_df.filter(
    F.col("turbine_name").isNull()
    | (F.trim(F.col("turbine_name")) == "")
).count()

print("DQ-03 - Missing turbine names:", missing_turbine_count)
print("Result:", "PASS" if missing_turbine_count == 0 else "FAIL")

In [0]:
# DQ-04: Check for negative energy production

negative_energy_count = silver_df.filter(
    F.col("energy_produced") < 0
).count()

print("DQ-04 - Negative energy values:", negative_energy_count)
print("Result:", "PASS" if negative_energy_count == 0 else "FAIL")


# DQ-05: Check for negative wind speed

negative_wind_count = silver_df.filter(
    F.col("wind_speed") < 0
).count()

print("DQ-05 - Negative wind speed values:", negative_wind_count)
print("Result:", "PASS" if negative_wind_count == 0 else "FAIL")


# DQ-06: Check for unexpected statuses

allowed_statuses = [
    "Online",
    "Offline",
    "Preventive Maintenance",
    "Reactive Maintenance",
    "Fault",
    "Inspection"
]

invalid_status_count = silver_df.filter(
    F.col("status").isNull()
    | ~F.col("status").isin(allowed_statuses)
).count()

print("DQ-06 - Invalid or missing statuses:", invalid_status_count)
print("Result:", "PASS" if invalid_status_count == 0 else "FAIL")

In [0]:
# DQ-07: Check for missing measurement timestamps

missing_timestamp_count = silver_df.filter(
    F.col("measurement_timestamp").isNull()
).count()

print("DQ-07 - Missing timestamps:", missing_timestamp_count)
print(
    "Result:",
    "PASS" if missing_timestamp_count == 0 else "FAIL"
)

In [0]:
# DQ-08: Check for duplicated turbine measurements

duplicate_measurement_count = (
    silver_df
    .groupBy("turbine_name", "measurement_timestamp")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "DQ-08 - Duplicated turbine measurements:",
    duplicate_measurement_count
)

print(
    "Result:",
    "PASS" if duplicate_measurement_count == 0 else "FAIL"
)

In [0]:
# Data Quality Report

quality_results = [
    {
        "rule_id": "DQ-01",
        "rule_description": "production_id must not be null",
        "invalid_records": missing_id_count
    },
    {
        "rule_id": "DQ-02",
        "rule_description": "production_id must be unique",
        "invalid_records": duplicate_id_count
    },
    {
        "rule_id": "DQ-03",
        "rule_description": "turbine_name must not be null or empty",
        "invalid_records": missing_turbine_count
    },
    {
        "rule_id": "DQ-04",
        "rule_description": "energy_produced must be non-negative",
        "invalid_records": negative_energy_count
    },
    {
        "rule_id": "DQ-05",
        "rule_description": "wind_speed must be non-negative",
        "invalid_records": negative_wind_count
    },
    {
        "rule_id": "DQ-06",
        "rule_description": "status must belong to the approved list",
        "invalid_records": invalid_status_count
    },
    {
        "rule_id": "DQ-07",
        "rule_description": "measurement_timestamp must not be null",
        "invalid_records": missing_timestamp_count
    },
    {
        "rule_id": "DQ-08",
        "rule_description": "turbine and timestamp combination must be unique",
        "invalid_records": duplicate_measurement_count
    }
]

quality_report = spark.createDataFrame(quality_results)

quality_report = quality_report.withColumn(
    "result",
    F.when(F.col("invalid_records") == 0, "PASS")
     .otherwise("FAIL")
)

display(quality_report.orderBy("rule_id"))

In [0]:
# Data Quality Summary

total_rules = quality_report.count()

passed_rules = quality_report.filter(
    F.col("result") == "PASS"
).count()

failed_rules = quality_report.filter(
    F.col("result") == "FAIL"
).count()

print("Total rules:", total_rules)
print("Passed rules:", passed_rules)
print("Failed rules:", failed_rules)

if failed_rules == 0:
    print("Overall quality status: PASS")
else:
    print("Overall quality status: FAIL")

In [0]:
# Save the data quality report as a Delta table

quality_report.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.data_quality_results")

print("Data quality report saved successfully.")

In [0]:
# Read the saved data quality report

saved_quality_report = spark.table(
    "workspace.default.data_quality_results"
)

print("Number of rows:", saved_quality_report.count())
print("Number of columns:", len(saved_quality_report.columns))

display(saved_quality_report.orderBy("rule_id"))

In [0]:
# Add execution timestamp to the data quality report

quality_report_with_timestamp = quality_report.withColumn(
    "execution_timestamp",
    F.current_timestamp()
)

display(quality_report_with_timestamp.orderBy("rule_id"))

In [0]:
# Append the timestamped data quality results

quality_report_with_timestamp.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.data_quality_results")

print("Timestamped data quality results appended successfully.")

In [0]:
%sql
ALTER TABLE workspace.default.data_quality_results
ADD COLUMNS (execution_timestamp TIMESTAMP);

In [0]:
quality_report_with_timestamp.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.data_quality_results")

print("Timestamped data quality results appended successfully.")

In [0]:
quality_history = spark.table(
    "workspace.default.data_quality_results"
)

print("Total records:", quality_history.count())

display(
    quality_history
    .orderBy("execution_timestamp", "rule_id")
)

In [0]:
def check_missing_production_id(df):
    return df.filter(
        F.col("production_id").isNull()
    ).count()

In [0]:
missing_id_count = check_missing_production_id(silver_df)
print("DQ-01 - Missing production IDs:", missing_id_count)

In [0]:
def run_quality_checks(df):

    # DQ-01: Check for missing production identifiers
    missing_id_count = df.filter(
        F.col("production_id").isNull()
    ).count()

    # DQ-02: Check for duplicated production identifiers
    duplicate_id_count = (
        df
        .groupBy("production_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    # DQ-03: Check for missing turbine names
    missing_turbine_count = df.filter(
        F.col("turbine_name").isNull()
        | (F.trim(F.col("turbine_name")) == "")
    ).count()

    # DQ-04: Check for negative energy production
    negative_energy_count = df.filter(
        F.col("energy_produced") < 0
    ).count()

    # DQ-05: Check for negative wind speed
    negative_wind_count = df.filter(
        F.col("wind_speed") < 0
    ).count()

    # DQ-06: Check for unexpected statuses
    allowed_statuses = [
        "Online",
        "Offline",
        "Preventive Maintenance",
        "Reactive Maintenance",
        "Fault",
        "Inspection"
    ]

    invalid_status_count = df.filter(
        F.col("status").isNull()
        | ~F.col("status").isin(allowed_statuses)
    ).count()

    # DQ-07: Check for missing measurement timestamps
    missing_timestamp_count = df.filter(
        F.col("measurement_timestamp").isNull()
    ).count()

    # DQ-08: Check for duplicated turbine measurements
    duplicate_measurement_count = (
        df
        .groupBy("turbine_name", "measurement_timestamp")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    # Collect the results of all eight rules
    quality_results = [
        {
            "rule_id": "DQ-01",
            "rule_description": "production_id must not be null",
            "invalid_records": missing_id_count
        },
        {
            "rule_id": "DQ-02",
            "rule_description": "production_id must be unique",
            "invalid_records": duplicate_id_count
        },
        {
            "rule_id": "DQ-03",
            "rule_description": "turbine_name must not be null or empty",
            "invalid_records": missing_turbine_count
        },
        {
            "rule_id": "DQ-04",
            "rule_description": "energy_produced must be non-negative",
            "invalid_records": negative_energy_count
        },
        {
            "rule_id": "DQ-05",
            "rule_description": "wind_speed must be non-negative",
            "invalid_records": negative_wind_count
        },
        {
            "rule_id": "DQ-06",
            "rule_description": "status must belong to the approved list",
            "invalid_records": invalid_status_count
        },
        {
            "rule_id": "DQ-07",
            "rule_description": "measurement_timestamp must not be null",
            "invalid_records": missing_timestamp_count
        },
        {
            "rule_id": "DQ-08",
            "rule_description": "turbine and timestamp combination must be unique",
            "invalid_records": duplicate_measurement_count
        }
    ]

    # Create the quality report
    quality_report = spark.createDataFrame(quality_results)

    # Assign PASS or FAIL to each rule
    quality_report = quality_report.withColumn(
        "result",
        F.when(
            F.col("invalid_records") == 0,
            "PASS"
        ).otherwise("FAIL")
    )

    # Add the execution timestamp
    quality_report = quality_report.withColumn(
        "execution_timestamp",
        F.current_timestamp()
    )

    return quality_report

In [0]:
quality_report = run_quality_checks(silver_df)
display(quality_report)

In [0]:
quality_report = run_quality_checks(silver_df)

display(quality_report.orderBy("rule_id"))

In [0]:
quality_history_df = spark.table(
    "workspace.default.data_quality_results"
)

print("Nombre total de résultats :", quality_history_df.count())

display(
    quality_history_df
    .orderBy("execution_timestamp", "rule_id")
)

In [0]:
from pyspark.sql import functions as F

quality_history_df = spark.table(
    "workspace.default.data_quality_results"
)

display(
    quality_history_df
    .groupBy("execution_timestamp")
    .agg(
        F.count("*").alias("number_of_rules"),
        F.sum(
            F.when(F.col("result") == "PASS", 1).otherwise(0)
        ).alias("passed_rules"),
        F.sum(
            F.when(F.col("result") == "FAIL", 1).otherwise(0)
        ).alias("failed_rules")
    )
    .orderBy("execution_timestamp")
)

In [0]:
print("Nombre de lignes Silver :", silver_df.count())

print("Nombre de colonnes :", len(silver_df.columns))

print("Colonnes disponibles :")
print(silver_df.columns)

In [0]:
from pyspark.sql import functions as F

# DQ-01: Missing production ID
missing_id = F.col("production_id").isNull()

# DQ-03: Missing or empty turbine name
missing_turbine = (
    F.col("turbine_name").isNull()
    | (F.trim(F.col("turbine_name")) == "")
)

# DQ-04: Negative energy production
negative_energy = F.col("energy_produced") < 0

# DQ-05: Negative wind speed
negative_wind_speed = F.col("wind_speed") < 0

# DQ-06: Invalid status
allowed_statuses = [
    "Online",
    "Offline",
    "Preventive Maintenance",
    "Reactive Maintenance",
    "Fault",
    "Inspection"
]

invalid_status = (
    F.col("status").isNull()
    | (~F.col("status").isin(allowed_statuses))
)

# DQ-07: Missing measurement timestamp
missing_timestamp = F.col("measurement_timestamp").isNull()

# Combine all row-level quality conditions
invalid_condition = (
    missing_id
    | missing_turbine
    | negative_energy
    | negative_wind_speed
    | invalid_status
    | missing_timestamp
)

# Keep records that violate at least one rule
invalid_silver_df = silver_df.filter(invalid_condition)

print("Total Silver records:", silver_df.count())
print("Invalid records detected:", invalid_silver_df.count())

In [0]:
# DQ-02: Identify duplicated production IDs

duplicate_ids_df = (
    silver_df
    .groupBy("production_id")
    .count()
    .filter(F.col("count") > 1)
    .select("production_id")
)

invalid_duplicate_ids_df = silver_df.join(
    duplicate_ids_df,
    on="production_id",
    how="inner"
)

print(
    "Number of duplicated production IDs:",
    duplicate_ids_df.count()
)

print(
    "Number of records affected by duplicate IDs:",
    invalid_duplicate_ids_df.count()
)

display(invalid_duplicate_ids_df)

In [0]:
# DQ-08: Identify duplicated turbine/timestamp combinations

duplicate_measurements_df = (
    silver_df
    .groupBy("turbine_name", "measurement_timestamp")
    .count()
    .filter(F.col("count") > 1)
    .select("turbine_name", "measurement_timestamp")
)

invalid_duplicate_measurements_df = silver_df.join(
    duplicate_measurements_df,
    on=["turbine_name", "measurement_timestamp"],
    how="inner"
)

print(
    "Number of duplicated turbine/timestamp combinations:",
    duplicate_measurements_df.count()
)

print(
    "Number of records affected by duplicate measurements:",
    invalid_duplicate_measurements_df.count()
)

display(invalid_duplicate_measurements_df)

In [0]:
from pyspark.sql import functions as F

# DQ-01: Missing production ID
missing_id = F.col("production_id").isNull()

# DQ-03: Missing or empty turbine name
missing_turbine = (
    F.col("turbine_name").isNull()
    | (F.trim(F.col("turbine_name")) == "")
)

# DQ-04: Negative energy production
negative_energy = F.col("energy_produced") < 0

# DQ-05: Negative wind speed
negative_wind_speed = F.col("wind_speed") < 0

# DQ-06: Invalid status
allowed_statuses = [
    "Online",
    "Offline",
    "Preventive Maintenance",
    "Reactive Maintenance",
    "Fault",
    "Inspection"
]

invalid_status = (
    F.col("status").isNull()
    | (~F.col("status").isin(allowed_statuses))
)

# DQ-07: Missing measurement timestamp
missing_timestamp = F.col("measurement_timestamp").isNull()

# Combine the six row-level conditions
invalid_condition = (
    missing_id
    | missing_turbine
    | negative_energy
    | negative_wind_speed
    | invalid_status
    | missing_timestamp
)

print("The six row-level quality conditions are ready.")

In [0]:
# DQ-02: Join records with duplicated production IDs
invalid_duplicate_ids_df = silver_df.join(
    duplicate_ids_df,
    on="production_id",
    how="inner"
)

# DQ-08: Join records with duplicated turbine/timestamp combinations
invalid_duplicate_measurements_df = silver_df.join(
    duplicate_measurements_df,
    on=["turbine_name", "measurement_timestamp"],
    how="inner"
)

# Get the keys of all records violating DQ-02 or DQ-08
duplicate_id_keys_df = duplicate_ids_df.select("production_id")

duplicate_measurement_keys_df = duplicate_measurements_df.select(
    "turbine_name",
    "measurement_timestamp"
)

# Identify all records violating at least one quality rule
invalid_row_level_df = silver_df.filter(invalid_condition)

invalid_id_records_df = silver_df.join(
    duplicate_id_keys_df,
    on="production_id",
    how="left_semi"
)

invalid_measurement_records_df = silver_df.join(
    duplicate_measurement_keys_df,
    on=["turbine_name", "measurement_timestamp"],
    how="left_semi"
)

# Combine all invalid records and remove repeated copies
quarantine_df = (
    invalid_row_level_df
    .unionByName(invalid_id_records_df)
    .unionByName(invalid_measurement_records_df)
    .dropDuplicates()
)

# Keep records that do not violate any of the eight rules
invalid_keys_df = (
    invalid_id_records_df.select("production_id")
    .union(invalid_measurement_records_df.select("production_id"))
    .union(invalid_row_level_df.select("production_id"))
    .distinct()
)

silver_validated_df = silver_df.join(
    invalid_keys_df,
    on="production_id",
    how="left_anti"
)

print("Original Silver records:", silver_df.count())
print("Records in quarantine:", quarantine_df.count())
print("Validated Silver records:", silver_validated_df.count())

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

test_data = [
    Row(
        production_id=9001,
        turbine_name="Turbine A",
        energy_produced=150.0
    ),
    Row(
        production_id=9002,
        turbine_name="Turbine B",
        energy_produced=-25.0
    ),
    Row(
        production_id=9003,
        turbine_name=None,
        energy_produced=120.0
    )
]

test_df = spark.createDataFrame(test_data)

display(test_df)

In [0]:
# Règle 1 : l'énergie produite ne doit pas être négative
negative_energy = F.col("energy_produced") < 0

# Règle 2 : le nom de l'éolienne ne doit pas être nul ou vide
missing_turbine = (
    F.col("turbine_name").isNull()
    | (F.trim(F.col("turbine_name")) == "")
)

# Une ligne est invalide si au moins une règle n'est pas respectée
invalid_condition = negative_energy | missing_turbine

# Séparer les données invalides des données conformes
test_quarantine_df = test_df.filter(invalid_condition)
test_validated_df = test_df.filter(~invalid_condition)

print("Nombre de lignes de test :", test_df.count())
print("Nombre de lignes en quarantaine :", test_quarantine_df.count())
print("Nombre de lignes valides :", test_validated_df.count())

display(test_quarantine_df)
display(test_validated_df)

In [0]:
# Afficher les données mises en quarantaine
print("=== Données en quarantaine ===")
display(test_quarantine_df)

# Afficher les données validées
print("=== Données validées ===")
display(test_validated_df)

# Vérifier les identifiants détectés
print("Identifiants des lignes invalides :")
display(test_quarantine_df.select("production_id"))

print("Identifiants des lignes valides :")
display(test_validated_df.select("production_id"))

In [0]:
quality_history_df = spark.table(
    "workspace.default.data_quality_results"
)

print("Total quality results:", quality_history_df.count())

display(
    quality_history_df
    .groupBy("execution_timestamp")
    .agg(
        F.count("*").alias("number_of_rules"),
        F.sum(
            F.when(F.col("result") == "PASS", 1).otherwise(0)
        ).alias("passed_rules"),
        F.sum(
            F.when(F.col("result") == "FAIL", 1).otherwise(0)
        ).alias("failed_rules")
    )
    .orderBy(F.col("execution_timestamp").desc_nulls_last())
)